# Encoder–Ridge with quadratic terms

Self-contained preprocessing and features copied from **features_engineering.ipynb**.
All feature groups are combined, including the observed-OFF features; patient medians,
disease duration and LEDD × delay / LEDD-to-delay interactions are enabled.
Ridge receives these features and one contextual embedding per visit. Each input is kept
alongside its square, including embedding coordinates, then standardized before regression.
Optional final predictions use Ridge with the best alpha from its search.

The encoder receives the original imputed features, their missingness indicators and elapsed
time since the first visit. It uses two bidirectional Transformer encoder blocks, four heads,
width 64 and **no CLS token**. Each patient's available visits form one sequence.

Validation trains the whole pipeline separately in each patient split. There are **no
out-of-fold embeddings**. The outer validation patients never train the encoder or Ridge.
An internal patient holdout selects the epoch count; the encoder is then refitted on all
outer training patients for that many epochs.


## 1. Local preprocessing, features and training options

History helpers, predictive imputation and all engineered features are defined here; no cells
are imported from another notebook. Group comparisons, plots and HistGB tuning/submission
from the source notebook are omitted. All preprocessing is fitted inside each patient split.
PyTorch and Optuna are required.


In [11]:
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import optuna
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "data/X_train_6ZIKlTY.csv").is_file())
DATA_DIR = PROJECT_DIR / "data"
X = pd.read_csv(DATA_DIR / "X_train_6ZIKlTY.csv")
Y = pd.read_csv(DATA_DIR / "y_train_lXj6X5y.csv")
assert X["Index"].equals(Y["Index"])
y = Y["target"]

# Score/dose summaries exclude the current visit.
diagnosis_history = [
    "ledd_first_observed", "time_to_first_observed_ledd", "time_since_first_visit",
]
history_features = diagnosis_history + [
    f"{col}_{stat}_patient" for col in ["on", "off", "ledd"] for stat in ["median", "slope"]
] + ["disease_duration"]

def median_other_visits(s):
    values = np.tile(s.to_numpy(), (len(s), 1))
    np.fill_diagonal(values, np.nan)
    available = ~np.isnan(values).all(axis=1)
    medians = np.full(len(s), np.nan)
    medians[available] = np.nanmedian(values[available], axis=1)
    return medians

def add_patient_history(df):
    visits = df.sort_values(["patient_id", "age"])
    first_age = visits.groupby("patient_id")["age"].min()
    first_dose = visits.dropna(subset=["ledd"]).drop_duplicates("patient_id").set_index("patient_id")
    out = df.copy()
    out["ledd_first_observed"] = out["patient_id"].map(first_dose["ledd"])
    out["time_to_first_observed_ledd"] = out["patient_id"].map(first_dose["age"]) - out["patient_id"].map(first_age)
    out["time_since_first_visit"] = out["age"] - out["patient_id"].map(first_age)
    out["disease_duration"] = out["age"] - out["age_at_diagnosis"]
    t = out["time_since_first_visit"]
    for col in ["on", "off", "ledd"]:
        v = df[col]
        out[f"{col}_median_patient"] = df.groupby("patient_id")[col].transform(median_other_visits)
        # Subtract this visit's contributions from the patient's regression sums.
        sums = pd.DataFrame({"n": v.notna().astype(int), "t": t.where(v.notna(), 0),
                             "v": v.fillna(0), "tt": (t*t).where(v.notna(), 0), "tv": (t*v).fillna(0)})
        others = sums.groupby(df["patient_id"]).transform("sum") - sums
        denominator = others["n"] * others["tt"] - others["t"]**2
        out[f"{col}_slope_patient"] = (
            (others["n"] * others["tv"] - others["t"] * others["v"]) / denominator
        ).where((others["n"] >= 3) & (denominator > 1e-12))
    return out

# Keep patient_id through imputation for patient histories and encoder sequences.
encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["cohort", "gene"])],
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")
preprocessor = make_pipeline(
    FunctionTransformer(add_patient_history), encoder,
    FunctionTransformer(lambda df: df.drop(columns="Index")),
)

predictors_history = {
    "age_at_diagnosis": ["age"] + diagnosis_history,
    "on": ["off", "age_at_diagnosis", "on_median_patient",
           "off_median_patient", "ledd_median_patient", "disease_duration",
           "time_since_first_visit", "on_slope_patient"],
    "off": ["on", "age_at_diagnosis", "time_since_intake_on", "on_median_patient",
            "off_median_patient", "ledd_median_patient",
            "time_since_first_visit", "off_slope_patient"],
    "ledd": ["age", "age_at_diagnosis", "on", "cohort_A",
             "off_median_patient", "ledd_median_patient", "disease_duration",
             "time_since_first_visit", "ledd_slope_patient"],
}

class PredictiveImputer(TransformerMixin, BaseEstimator):
    def __init__(self, predictors):
        self.predictors = predictors

    def fit(self, X, y=None):
        self.medians_, self.models_ = X.median(numeric_only=True), {}
        self.indicator_ = MissingIndicator(error_on_new=False).set_output(
            transform="pandas"
        ).fit(X.drop(columns=history_features))
        for col, features in self.predictors.items():
            observed = X[col].notna()
            self.models_[col] = HistGradientBoostingRegressor(
                max_iter=50, max_leaf_nodes=15, random_state=42
            ).fit(X.loc[observed, features], X.loc[observed, col])
        return self

    def transform(self, X):
        filled = X.copy()
        for col, features in self.predictors.items():
            missing = X[col].isna()
            if missing.any():
                values = self.models_[col].predict(X.loc[missing, features])
                if col == "age_at_diagnosis":  # Diagnosis cannot follow this visit.
                    values = values.clip(max=X.loc[missing, "age"].to_numpy())
                filled.loc[missing, col] = values
        if "disease_duration" in filled:
            filled["disease_duration"] = filled["age"] - filled["age_at_diagnosis"]
        # History supports imputation only; indicators describe the original missingness.
        values = filled.fillna(self.medians_).drop(columns=history_features)
        indicators = self.indicator_.transform(X.drop(columns=history_features)).astype(float)
        return pd.concat([values, indicators], axis=1)

def build_visit_features(data):
    patients = data.groupby("patient_id")
    ages = patients["age"]
    features = pd.DataFrame(index=data.index)
    features["time_since_first_visit"] = data["age"] - ages.transform("min")
    features["n_past_visits"] = (ages.rank(method="min") - 1).astype(int)
    features["n_future_visits"] = (ages.transform("size") - ages.rank(method="max")).astype(int)
    for col in ["on", "off", "ledd"]:
        values = data[col]
        features[f"{col}_median_patient"] = patients[col].transform("median")
        features[f"{col}_n_observed"] = patients[col].transform("count")
        age = data["age"].where(values.notna())
        dt = age - age.groupby(data["patient_id"]).transform("mean")
        dv = values - patients[col].transform("mean")
        covariance = (dt * dv).groupby(data["patient_id"]).transform("sum")
        variance = (dt**2).groupby(data["patient_id"]).transform("sum")
        features[f"{col}_slope_patient"] = (covariance / variance).where(variance > 1e-12)
    gap = data["off"] - data["on"]
    features["off_on_median_patient"] = gap.groupby(data["patient_id"]).transform("median")
    features["disease_duration"] = data["age"] - data["age_at_diagnosis"]
    for score in ["on", "off"]:
        delay = data[f"time_since_intake_{score}"]
        features[f"ledd_x_time_{score}"] = data["ledd"] * delay
        features[f"ledd_time_ratio_{score}"] = data["ledd"] / (1 + delay)
    visits = data.sort_values(["patient_id", "age"])
    for col in ["on", "off"]:
        mean_age = data["age"].where(data[col].notna()).groupby(data["patient_id"]).transform("mean")
        features[f"{col}_trend_at_visit"] = patients[col].transform("mean") + (
            features[f"{col}_slope_patient"] * (data["age"] - mean_age)
        )
        features[f"{col}_std_patient"] = patients[col].transform("std")
        observed = visits[[col, "age"]].where(visits[col].notna(), axis=0)
        for label, shift, fill in [("previous", 1, "ffill"), ("next", -1, "bfill")]:
            neighbor = getattr(observed.groupby(visits["patient_id"]), fill)()
            neighbor = neighbor.groupby(visits["patient_id"]).shift(shift)
            features[f"{col}_{label}"] = neighbor[col]
            features[f"{col}_{label}_gap"] = (visits["age"] - neighbor["age"]).abs()

    # Use only observed OFF measurements for these two features.
    off = data["off"].mask(data["missingindicator_off"].astype(bool)) if "missingindicator_off" in data else data["off"]
    t = data["age"] - ages.transform("mean")
    sums = pd.DataFrame({"n": off.notna().astype(int), "t": t.where(off.notna(), 0),
                         "v": off.fillna(0), "tt": (t*t).where(off.notna(), 0), "tv": (t*off).fillna(0)})
    others = sums.groupby(data["patient_id"]).transform("sum") - sums
    denominator = others["n"] * others["tt"] - others["t"]**2
    slope = ((others["n"] * others["tv"] - others["t"] * others["v"])
             / denominator).where(denominator > others["n"] * 1e-10, 0)
    features["off_trend_other_visits"] = (
        others["v"] / others["n"] + slope * (t - others["t"] / others["n"])
    )
    features["off_observed_fraction_patient"] = off.notna().groupby(data["patient_id"]).transform("mean")
    return features

baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("imputation", PredictiveImputer(predictors_history)),
    ("features_eng", FunctionTransformer(
        lambda df: df.join(build_visit_features(df)).drop(columns="patient_id")
    )),
    ("feature_imputation", SimpleImputer(
        strategy="median", add_indicator=True, keep_empty_features=True
    ).set_output(transform="pandas")),
    ("model", make_pipeline(
        FunctionTransformer(lambda df: pd.concat([df, df.pow(2).add_suffix("_squared")], axis=1)),
        StandardScaler(), Ridge(),
    )),
])

ENCODER = dict(width=64, heads=4, layers=2, dropout=0.1)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function <lambda> at 0x1265456c0>)),
                ('standardscaler', StandardScaler()), ('ridge', Ridge())])


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [12]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Small bidirectional encoder

Dense input projection → two sequential TransformerEncoder layers → one embedding per visit.
Each layer already includes parallel attention heads, feed-forward layers, residual connections
and LayerNorm. The regression head is used for supervised training only.

In [13]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=2 * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        self.head = nn.Linear(width, 1)

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [14]:
def train_network(training, seed, epochs, validation=None):
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; Ridge receives engineered features plus
contextual embeddings, with no CLS token, pooling or predicted-score column.


In [15]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.head.in_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Fold-specific Encoder–Ridge validation

Use the same five patient-separated splits as the existing notebooks. Preprocessing is fitted
again in each split and shared by the baseline and augmented model. For each seed, train a
fresh encoder on that split's training patients, freeze it, append its embeddings and fit
Ridge with linear and squared inputs, standardization and default alpha=1. Cache the features for the small search below.
Negative delta_MSE means the embeddings help. The standard deviation describes variability
across splits and seeds, not uncertainty about the true test performance.


In [16]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = []
tuning_folds = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        imputed_train = prep[:2].transform(raw_train)
        imputed_val = prep[:2].transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        baseline_mse = mean_squared_error(yv, original.predict(validation))

        for seed in SEEDS:
            fitted_encoder = fit_encoder(raw_train, yt, imputed_train, seed)
            z_train, _ = encoder_outputs(fitted_encoder, raw_train, imputed_train)
            z_val, encoder_prediction = encoder_outputs(fitted_encoder, raw_val, imputed_val)
            tuning_folds.append((train.join(z_train), validation.join(z_val), yt, yv))
            augmented = clone(baseline[-1]).fit(train.join(z_train), yt)
            mse = mean_squared_error(yv, augmented.predict(validation.join(z_val)))
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"],
                               Ridge=baseline_mse, Transformer=mean_squared_error(yv, encoder_prediction),
                               Encoder_Ridge=mse, delta_MSE=mse - baseline_mse))
            print(f"Fold {fold}/5, seed {seed}, {fitted_encoder['epochs']} epochs: "
                  f"Ridge {baseline_mse:.4f} → Encoder–Ridge {mse:.4f}")
scores = pd.DataFrame(scores)
scores

Fold 1/5, seed 42, 63 epochs: Ridge 16.2936 → Encoder–Ridge 7.7593
Fold 1/5, seed 43, 58 epochs: Ridge 16.2936 → Encoder–Ridge 7.4072
Fold 1/5, seed 44, 52 epochs: Ridge 16.2936 → Encoder–Ridge 7.6057
Fold 2/5, seed 42, 32 epochs: Ridge 15.5594 → Encoder–Ridge 8.3644
Fold 2/5, seed 43, 35 epochs: Ridge 15.5594 → Encoder–Ridge 7.7971
Fold 2/5, seed 44, 42 epochs: Ridge 15.5594 → Encoder–Ridge 7.8653
Fold 3/5, seed 42, 40 epochs: Ridge 15.3134 → Encoder–Ridge 7.8962
Fold 3/5, seed 43, 24 epochs: Ridge 15.3134 → Encoder–Ridge 8.0499
Fold 3/5, seed 44, 30 epochs: Ridge 15.3134 → Encoder–Ridge 8.0940
Fold 4/5, seed 42, 38 epochs: Ridge 16.0632 → Encoder–Ridge 8.0202
Fold 4/5, seed 43, 59 epochs: Ridge 16.0632 → Encoder–Ridge 7.4587
Fold 4/5, seed 44, 17 epochs: Ridge 16.0632 → Encoder–Ridge 8.4218
Fold 5/5, seed 42, 34 epochs: Ridge 16.2175 → Encoder–Ridge 8.0625
Fold 5/5, seed 43, 70 epochs: Ridge 16.2175 → Encoder–Ridge 7.4999
Fold 5/5, seed 44, 45 epochs: Ridge 16.2175 → Encoder–Ridge 7.

,fold,seed,epochs,Ridge,Transformer,Encoder_Ridge,delta_MSE
0,1,42,63,16.293577,8.251435,7.759285,-8.534292
1,1,43,58,16.293577,8.413365,7.407200,-8.886378
2,1,44,52,16.293577,8.343530,7.605669,-8.687909
3,2,42,32,15.559448,9.982853,8.364353,-7.195095
4,2,43,35,15.559448,9.403493,7.797083,-7.762365
5,2,44,42,15.559448,9.156487,7.865256,-7.694192
6,3,42,40,15.313380,9.498173,7.896185,-7.417195
7,3,43,24,15.313380,8.703908,8.049949,-7.263431
8,3,44,30,15.313380,9.220584,8.094009,-7.219371
9,4,42,38,16.063195,8.686321,8.020196,-8.042999


In [17]:
summary = scores[["Ridge", "Transformer", "Encoder_Ridge"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["Ridge", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
Ridge,15.889428,0.385203,3.986155,0.000000
Transformer,8.780511,0.602111,2.963193,44.739918
Encoder_Ridge,7.871945,0.295304,2.805699,50.457973


## 7. Small Optuna search for Ridge

Try 10 alpha values on the cached patient splits. Scaling is refitted on training rows only;
the encoder is not retrained during the search. This reuses the validation splits for tuning,
so the best MSE is a tuning score, not an independent performance estimate.


In [18]:
def ridge_objective(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 1e4, log=True)
    model = clone(baseline[-1]).set_params(ridge__alpha=alpha)
    return float(np.mean([
        mean_squared_error(yv, model.fit(xt, yt).predict(xv))
        for xt, xv, yt, yv in tuning_folds
    ]))


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=3))
study.enqueue_trial({"alpha": 1.0})
with threadpool_limits(limits=3):
    study.optimize(ridge_objective, n_trials=10)
best_ridge = clone(baseline[-1]).set_params(ridge__alpha=study.best_params["alpha"])
print(f"Best alpha: {study.best_params['alpha']:.4g}; tuning MSE: {study.best_value:.4f}")


[I 2026-11-03 10:39:35,908] A new study created in memory with name: no-name-cc04636a-1d9d-492c-b563-567ed9876215
[I 2026-11-03 10:39:36,896] Trial 0 finished with value: 7.8719448901861195 and parameters: {'alpha': 1.0}. Best is trial 0 with value: 7.8719448901861195.
[I 2026-11-03 10:39:37,640] Trial 1 finished with value: 7.872394323366454 and parameters: {'alpha': 0.41858227295469747}. Best is trial 0 with value: 7.8719448901861195.
[I 2026-11-03 10:39:38,364] Trial 2 finished with value: 7.866324353655156 and parameters: {'alpha': 4518.56095102411}. Best is trial 2 with value: 7.866324353655156.
[I 2026-11-03 10:39:39,189] Trial 3 finished with value: 7.904334018901928 and parameters: {'alpha': 6198.780154146059}. Best is trial 2 with value: 7.866324353655156.
[I 2026-11-03 10:39:39,947] Trial 4 finished with value: 7.872859857669593 and parameters: {'alpha': 0.0011014892948576012}. Best is trial 2 with value: 7.866324353655156.
[I 2026-11-03 10:39:40,709] Trial 5 finished with va

Best alpha: 95.72; tuning MSE: 7.8492


## 8. Optional final training and test predictions

After reviewing validation, set RUN_FINAL to True and choose a seed. A single encoder is
trained on all training patients and paired with quadratic Ridge using the best alpha from its search.
The outer test data is transformed only; it never fits preprocessing or either model.


In [19]:
RUN_FINAL = False
FINAL_SEED = 43

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    with threadpool_limits(limits=3):
        final_prep = clone(baseline[:-1])
        train = final_prep.fit_transform(X, y)
        test = final_prep.transform(X_test)
        imputed_train = final_prep[:2].transform(X)
        imputed_test = final_prep[:2].transform(X_test)
        final_encoder = fit_encoder(X, y, imputed_train, FINAL_SEED)
        z_train, _ = encoder_outputs(final_encoder, X, imputed_train)
        z_test, _ = encoder_outputs(final_encoder, X_test, imputed_test)
        final_ridge = clone(best_ridge).fit(train.join(z_train), y)
        prediction = final_ridge.predict(test.join(z_test))
    assert np.isfinite(prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": prediction})
    submission_path = DATA_DIR / "submission_encoder_ridge_quadratic_tuned.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")

## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Vary encoder depth, width and heads, or replace the encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed Ridge the encoder's final prediction instead of, or alongside, its embeddings.